<a href="https://colab.research.google.com/github/Emmanuel-Rono/Testing_Ai_Models/blob/codes_from_colab/LLM_Eval_with_Rogue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ROGUE = Recall-Oritneted Understudy for Gisting Evaluation

### This metric is useful in evaluatiing text summarization,machine translations, and other NLP task  especially when focusing on recall and ability to capture relevant information from the reference text.


In [ ]:
#Install rougue-score library

!pip install rouge-score

In [ ]:
from rouge_score import rouge_scorer
from typing import List,Dict

### ▶ Calculate all Rouge scores (rouge1,rouge2,rouge3)

In [ ]:
from rouge_score import rouge_scorer
from typing import List,Dict

class RougeEvaluator:
    def __init__(self, metrics: List[str] = None, use_stemmer: bool = True):

        if metrics is None:
            metrics = ['rouge1', 'rouge2', 'rougeL']
        self.scorer = rouge_scorer.RougeScorer(metrics, use_stemmer=use_stemmer)
        self.metrics = metrics # Store metrics for later use

    def evaluate(self, references: List[str], candidates: List[str]) -> Dict[str, Dict[str, float]]:

        # Compute average ROUGE scores across multiple candidate-reference pairs.

        assert len(references) == len(candidates), "References and candidates must have the same length"

        # Initialize total_scores using the actual metrics
        total_scores = {m: {"precision": 0.0, "recall": 0.0, "f1": 0.0} for m in self.metrics}

        for ref, cand in zip(references, candidates):
            scores = self.scorer.score(ref, cand)
            for metric, score in scores.items():
                total_scores[metric]["precision"] += score.precision
                total_scores[metric]["recall"] += score.recall
                total_scores[metric]["f1"] += score.fmeasure # Changed from 'fmeasure' to 'f1'

        n = len(references)
        avg_scores = {
            m: {
                "precision": round(v["precision"] / n, 4),
                "recall": round(v["recall"] / n, 4),
                "f1": round(v["f1"] / n, 4), # Changed from 'fmeasure' to 'f1'
            }
            for m, v in total_scores.items()
        }
        return avg_scores

 #In action
if __name__ == "__main__":
    references = [
        "Register a new patient into the system with valid details",
        "Doctor updates the patient record with treatment notes"
    ]

    candidates = [
        "Check Details Add a new patient record with ",
        "Physician modifies the patient file with medical notes"
    ]

    evaluator = RougeEvaluator()
    results = evaluator.evaluate(references, candidates)

    print("ROUGE Evaluation Results:")
    for metric, scores in results.items():
        print(f"{metric}: P={scores['precision']}, R={scores['recall']}, F1={scores['f1']}")